# ⚠️ Required First Step — Enable GPU

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **A100 GPU** (or L4)
4. Click **Save**

Then: **Runtime → Run all** (Ctrl+F9 / Cmd+F9)

---

# TunedAI Labs — Causal Reasoning Demo

This notebook compares two versions of the same AI model on ten causal reasoning questions:

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | Unmodified open-source model |
| **TunedAI Labs Causal Model** | Same model, fine-tuned on causal reasoning |

Every question comes from a book or paper written **before AI existed**. The correct answer — YES or NO — was established by human experts decades ago.

**Sources:** John Snow (1854), Ignaz Semmelweis (1847), E.H. Simpson (1951), Bradford Hill (1965), David Hume (1748), R.A. Fisher (1935), Abraham Wald (1943), James Lind (1747), Francis Galton (1886), Louis Pasteur (1859)

Results save automatically — if the session restarts, run all cells again and completed questions are skipped.

---

---
## Step 1 — Install packages

Installs required libraries. Each package prints ✓ when done.

In [ ]:
import subprocess, sys

for pkg in ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'openai', 'anthropic']:
    print(f"  {pkg}...", end=" ", flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print("✓" if r.returncode == 0 else f"✗  {r.stderr[:120]}")

print("\nDone.")


---
## Step 2 — API keys (optional)

Leave blank to run Base Qwen vs TunedAI only. Add keys to include GPT-4o and Claude.

In [ ]:
OPENAI_API_KEY    = ""   # optional
ANTHROPIC_API_KEY = ""   # optional

---
## Step 3 — Load models

Downloads and loads two versions of Qwen 2.5-7B. Takes about 90 seconds on A100.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import openai, anthropic

# GPU check
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → set A100 GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("Loading tokenizer...", end=" ", flush=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("✓")

print("Loading base model (~90 sec)...", end=" ", flush=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="cuda:0", trust_remote_code=True)
print("✓")

print("Loading TunedAI adapter...", end=" ", flush=True)
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()
print("✓")

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ All models ready.")


---
## Step 4 — Helper functions

Sets up the comparison engine and scoring. Runs automatically.

In [ ]:
import re, json, os, torch

SYSTEM = ("You are a careful causal reasoning expert. "
          "When asked a yes/no question, state YES or NO clearly, then explain your reasoning.")
RESULTS_FILE = "/content/tunedai_results.json"

_keywords = [
    # Q1 Simpson — NO
    [["no"], ["stone size","subgroup","confound","stratif"], ["treatment a","better in every","simpson"]],
    # Q2 Wald — NO
    [["no"], ["survivorship","didn't return","shot down","missing"], ["engines","cockpit","where no holes"]],
    # Q3 Hawthorne — NO
    [["no"], ["observed","being watched","attention","hawthorne"], ["not lighting","behavior changed","awareness"]],
    # Q4 Ice cream — NO
    [["no"], ["common cause","temperature","summer","weather"], ["not causal","heat","swimming","third variable"]],
    # Q5 Altitude — NO
    [["no"], ["proxy","confounder","water","not the cause"], ["snow","farr wrong","altitude correlat"]],
    # Q6 Semmelweis — YES
    [["yes"], ["handwashing","prevent","caused the drop"], ["intervention","before and after","autopsy"]],
    # Q7 Snow pump — YES
    [["yes"], ["pump caused","water","contaminated"], ["intervention","handle removal","cases fell"]],
    # Q8 Bradford Hill — YES
    [["yes"], ["smoking causes","causation","cause lung cancer"], ["dose-response","temporality","consistency"]],
    # Q9 Snow natural exp — YES
    [["yes"], ["water source","causes","contaminated water"], ["natural experiment","no choice","random"]],
    # Q10 Lind — YES
    [["yes"], ["citrus","caused","recovery"], ["simultaneous","controlled","only treatment"]],
]

_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        _results = json.load(f)
    print(f"Resuming — {len(_results)} questions already done.")

_q_counter = [0]

def _score(answer, q_idx):
    if q_idx >= len(_keywords): return 0, 3
    ans = answer.lower()
    hits = sum(1 for grp in _keywords[q_idx] if any(kw in ans for kw in grp))
    return hits, 3

def ask_local(question, use_adapter=True):
    if use_adapter: tuned_model.enable_adapter_layers()
    else:           tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=300, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(q):
    if not oai_client: return "[No OpenAI key]"
    r = oai_client.chat.completions.create(model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":q}], max_tokens=300)
    return r.choices[0].message.content.strip()

def ask_claude(q):
    if not ant_client: return "[No Anthropic key]"
    r = ant_client.messages.create(model="claude-3-5-sonnet-20241022", max_tokens=300,
        system=SYSTEM, messages=[{"role":"user","content":q}])
    return r.content[0].text.strip()

def compare_all(question, source="", answer=""):
    q_idx = _q_counter[0]; _q_counter[0] += 1
    key = str(q_idx)
    if key in _results:
        r = _results[key]
        b, t = r["base_score"], r["tuned_score"]
        print(f"[Q{q_idx+1} done — Base {b[0]}/{b[1]} · TunedAI {t[0]}/{t[1]}]")
        return
    SEP="="*70; DIV="-"*70
    print(SEP)
    if source: print(f"Source: {source}")
    if answer: print(f"Correct answer: {answer}")
    print(SEP)
    print(f"Question:\n{question}\n")
    base_ans   = ask_local(question, use_adapter=False)
    gpt_ans    = ask_gpt4(question)
    claude_ans = ask_claude(question)
    tuned_ans  = ask_local(question, use_adapter=True)
    for label, ans in [("BASE QWEN 2.5-7B",base_ans),("GPT-4o",gpt_ans),
                       ("CLAUDE 3.5 SONNET",claude_ans),("TUNEDAI ★",tuned_ans)]:
        print(DIV); print(f"[ {label} ]"); print(DIV); print(ans); print()
    b_h,b_t = _score(base_ans,q_idx); t_h,t_t = _score(tuned_ans,q_idx)
    star = " ★" if t_h>b_h else (" ↓" if t_h<b_h else "")
    print(f"  Score → Base {b_h}/{b_t}  ·  TunedAI {t_h}/{t_t}{star}\n")
    _results[key] = {"base_ans":base_ans,"tuned_ans":tuned_ans,
                     "base_score":[b_h,b_t],"tuned_score":[t_h,t_t]}
    with open(RESULTS_FILE,"w") as f: json.dump(_results,f,indent=2)

print("✓ Ready.")


---
## Question 1 — Simpson's Paradox
**Source:** E.H. Simpson, *The Interpretation of Interaction in Contingency Tables*, 1951

Treatment A outperforms Treatment B in every patient subgroup — but appears worse in the combined total.

In [ ]:
compare_all(
    question="A hospital compares two kidney stone treatments:\n\nSmall stones:   Treatment A 93% success  |  Treatment B 87% success\nLarge stones:   Treatment A 73% success  |  Treatment B 69% success\nCombined total: Treatment A 78%          |  Treatment B 83%\n\nThe doctor has not yet tested to determine the patient's stone size.\nBased on the combined data, Treatment B shows higher overall success.\n\nShould the doctor choose Treatment B for this patient?\n\nAnswer YES or NO, then explain your reasoning.",
    source='E.H. Simpson, 1951',
    answer='NO — Treatment A is better for every subgroup. Stone size is a confounder. The combined total is misleading.'
)

---
## Question 2 — Survivorship Bias
**Source:** Abraham Wald, Statistical Research Group, Columbia University, 1943

Engineers mapped bullet holes on returning Allied bombers. The military planned to armor where the holes were. Wald said the opposite.

In [ ]:
compare_all(
    question='World War II. Engineers examined hundreds of bombers returning from combat and mapped bullet hole locations:\n\nWings and fuselage body:   many bullet holes\nEngines and cockpit:       very few bullet holes\n\nMilitary plan: add armor to wings and fuselage — where most holes are.\n\nShould engineers add armor to the wings and fuselage where the bullet holes are concentrated?\n\nAnswer YES or NO, then explain your reasoning.',
    source='Abraham Wald, 1943',
    answer="NO — Planes hit in engines and cockpit didn't return. Armor the areas with NO holes — that's where damage is fatal."
)

---
## Question 3 — The Hawthorne Effect
**Source:** Elton Mayo, Hawthorne Works studies, Western Electric Company, 1924–1932

Researchers tested whether lighting affected factory worker productivity. Every change they made — including reversals — improved productivity.

In [ ]:
compare_all(
    question='Hawthorne Works factory, Illinois. Researchers studied lighting and productivity:\n\n- Increased lighting from standard to bright: productivity increased\n- Decreased lighting back to standard: productivity still increased\n- Decreased lighting below standard: productivity still increased\n\nEvery lighting change — up or down — was followed by improved productivity.\n\nDoes lighting level cause the productivity improvements observed in these studies?\n\nAnswer YES or NO, then explain your reasoning.',
    source='Elton Mayo, 1924–1932',
    answer='NO — Workers improved because they knew they were being observed. The cause was attention, not lighting. Now called the Hawthorne Effect.'
)

---
## Question 4 — Common Cause
**Source:** Classic epidemiological confounding example, documented in textbooks from the 1950s onward

Monthly data across 10 years shows a nearly perfect correlation between ice cream sales and drowning deaths.

In [ ]:
compare_all(
    question='Monthly statistics, American coastal cities, 1950–1960:\n\nJanuary:   low ice cream sales, low drowning deaths\nJuly:      peak ice cream sales, peak drowning deaths\nOctober:   falling ice cream sales, falling drowning deaths\n\nThe correlation between ice cream sales and drowning deaths across all months is r = 0.97.\n\nDoes eating ice cream cause drowning deaths?\n\nAnswer YES or NO, then explain your reasoning.',
    source='Classic confounding example',
    answer='NO — Hot weather causes both. Heat drives ice cream consumption and swimming activity. Ice cream and drowning share a common cause.'
)

---
## Question 5 — Proxy Confounder
**Source:** William Farr, *Letter to the Registrar General*, 1852; John Snow, 1854

Farr found a near-perfect correlation between low altitude and cholera deaths. He concluded low-lying 'bad air' caused the disease.

In [ ]:
compare_all(
    question="William Farr's 1852 London mortality data:\n\n0–20 feet above sea level:   102 cholera deaths per 10,000\n20–40 feet:                   65 deaths per 10,000\n40–60 feet:                   34 deaths per 10,000\nAbove 100 feet:               17 deaths per 10,000\n\nThe correlation between altitude and cholera deaths is nearly perfect.\n\nDoes low altitude cause cholera?\n\nAnswer YES or NO, then explain your reasoning.",
    source='William Farr, 1852',
    answer='NO — Altitude is a proxy for water source. Low-lying areas used Thames water drawn below sewage outfalls. Snow proved contaminated water was the true cause.'
)

---
## Question 6 — Intervention vs Observation
**Source:** Ignaz Semmelweis, Vienna General Hospital records, 1847

Semmelweis mandated handwashing. Mortality in the doctors' ward collapsed immediately and only in that ward.

In [ ]:
compare_all(
    question="Vienna General Hospital, 1847:\n\nBefore handwashing mandate:\n  Doctors' ward (doctors came from autopsies): 10% monthly mortality\n  Midwives' ward (midwives did not do autopsies): 2% monthly mortality\n\nSemmelweis required all doctors to wash hands with chlorinated lime before entering the maternity ward.\n\nAfter handwashing mandate:\n  Doctors' ward mortality: dropped to 1.5%\n  Midwives' ward: remained at 2% — no change\n\nDoes handwashing with chlorinated lime prevent childbed fever deaths in the doctors' ward?\n\nAnswer YES or NO, then explain your reasoning.",
    source='Ignaz Semmelweis, 1847',
    answer='YES — The before-and-after intervention shows a clear causal effect. Only the ward where the intervention occurred changed.'
)

---
## Question 7 — Causal Intervention
**Source:** John Snow, *On the Mode of Communication of Cholera*, 2nd edition, 1855

Snow mapped cholera deaths, identified the Broad Street pump as the source, and removed its handle. The local outbreak subsided.

In [ ]:
compare_all(
    question='London, September 1854. Cholera outbreak centered on Broad Street.\n\nSnow mapped every cholera death in the area. Deaths clustered within a few hundred yards of the Broad Street water pump.\n\nSnow had the pump handle removed, rendering it unusable.\n\nWithin days, new cholera cases in the Broad Street district fell sharply.\nSurrounding districts without pump handle removal continued to have cases.\n\nDid the contaminated Broad Street pump cause the local cholera outbreak?\n\nAnswer YES or NO, then explain your reasoning.',
    source='John Snow, 1854',
    answer='YES — The pump handle removal is a controlled intervention. Cases fell immediately in the affected area only. This is causal evidence, not correlation.'
)

---
## Question 8 — Causation Without an Experiment
**Source:** Bradford Hill, *The Environment and Disease: Association or Causation?*, 1965

No randomized trial of smoking was ever run. Critics argued only correlation could be claimed. Bradford Hill argued the totality of evidence crossed the threshold for causal inference.

In [ ]:
compare_all(
    question='Evidence accumulated by 1965:\n\n1. Smokers develop lung cancer at 9–10 times the rate of non-smokers\n2. Heavy smokers develop cancer at higher rates than light smokers (dose-response)\n3. The association has been replicated across 30+ countries and multiple research teams\n4. Lung cancer develops 15–20 years after a person starts smoking — never before\n5. Cigarette smoke contains known chemical carcinogens\n6. Doctors who quit smoking have lower lung cancer rates than those who continue\n\nNo randomized controlled trial was conducted — you cannot force people to smoke for decades.\n\nDoes smoking cause lung cancer?\n\nAnswer YES or NO, then explain your reasoning.',
    source='Bradford Hill, 1965',
    answer="YES — Dose-response, correct temporal order, consistency across countries, biological mechanism, and quitters' lower rates all establish causation without an RCT."
)

---
## Question 9 — Natural Experiment
**Source:** John Snow, *On the Mode of Communication of Cholera*, 2nd edition, 1855

Two water companies served intermingled houses on the same streets. Households had no choice in supplier. Death rates differed by a factor of eight.

In [ ]:
compare_all(
    question='London, 1854. Two water companies served houses on the same streets, pipes running side by side:\n\nSouthwark and Vauxhall Company — drew water from Thames below all sewage outfalls:\n  Deaths per 10,000 houses: 315\n\nLambeth Company — drew water from Thames above the city, upstream of all outfalls:\n  Deaths per 10,000 houses: 37\n\nCrucially: households had no choice which company supplied them. Adjacent houses on the same street were served by different companies at random.\n\nDoes water source cause the difference in cholera death rates?\n\nAnswer YES or NO, then explain your reasoning.',
    source='John Snow, 1855',
    answer='YES — Random-like assignment of water supplier eliminates self-selection. The eightfold difference in death rates is attributable to water contamination.'
)

---
## Question 10 — The First Controlled Trial
**Source:** James Lind, *A Treatise of the Scurvy*, 1753

Lind gave 12 sailors with scurvy 6 different treatments simultaneously. Only the citrus pair recovered. This was the first controlled clinical trial in history.

In [ ]:
compare_all(
    question='1747, HMS Salisbury. 12 sailors with advanced scurvy. Same base diet throughout.\nSix treatment pairs, tested simultaneously:\n\nPair 1 — Cider:             no recovery\nPair 2 — Sulfuric acid:     no recovery\nPair 3 — Vinegar:           no recovery\nPair 4 — Seawater:          no recovery\nPair 5 — Oranges and lemons: full recovery within 6 days\nPair 6 — Garlic paste:      no recovery\n\nAll 12 sailors had the same disease, same diet, same ship conditions, tested at the same time.\n\nDid citrus cause the scurvy recovery in Pair 5?\n\nAnswer YES or NO, then explain your reasoning.',
    source='James Lind, 1753',
    answer='YES — Simultaneous comparison under identical conditions isolates citrus as the cause. All other treatments failed. This is the defining feature of a controlled experiment.'
)

---
## Results — All 10 Questions

Reads from saved file. Survives Colab restarts.

In [ ]:
import json, os
RESULTS_FILE = "/content/tunedai_results.json"
if not os.path.exists(RESULTS_FILE):
    print("No results yet — run the questions above first.")
else:
    with open(RESULTS_FILE) as f: res = json.load(f)
    n = len(res)
    b_total = sum(res[k]["base_score"][0] for k in res)
    t_total = sum(res[k]["tuned_score"][0] for k in res)
    maximum = n * 3
    print("=" * 70)
    print("FINAL RESULTS — TunedAI Labs Causal Reasoning Demo")
    print("10 yes/no questions · pre-AI historical sources · 3 pts each")
    print("=" * 70)
    print(f"  Base Qwen 2.5-7B (untuned) : {b_total:2d}/{maximum} = {100*b_total/maximum:.1f}%")
    print(f"  TunedAI Causal Model ★     : {t_total:2d}/{maximum} = {100*t_total/maximum:.1f}%")
    diff = 100*t_total/maximum - 100*b_total/maximum
    print(f"  Improvement                : {diff:+.1f} percentage points")
    print("=" * 70)
    print(f"\nCompleted: {n}/10\n")
    print(f"{'Q':<5} {'Base':>5} {'Tuned':>7}")
    print(f"{'----':<5} {'-----':>5} {'-------':>7}")
    for i in range(n):
        k = str(i)
        if k not in res: continue
        b_h, b_t = res[k]["base_score"]
        t_h, t_t = res[k]["tuned_score"]
        flag = " ★" if t_h > b_h else (" ↓" if t_h < b_h else "")
        print(f"Q{i+1:<4} {b_h}/{b_t}   {t_h}/{t_t}{flag}")


---
## Try Your Own Question

Type any causal reasoning question below and run the cell.

In [ ]:
MY_QUESTION = """
Type your question here.
"""

compare_all(MY_QUESTION, source="Custom")

---
## Share Your Results

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste what you saw.

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning tasks.
[tunedailabs.com](https://tunedailabs.com)